# Main追加実験 FとB50
既存bank・モデルを保持して、旧方式の未実施ロールアウト、F予算1.5/B50の独立学習、D50を含むON/OFF評価をWindowsで直列実行します。旧FはHBAとして扱います。上から実行するとdevelopmentと正規Fの確率評価まで進みます。decision全方式と残りの校正は任意です。

In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from tqdm.notebook import tqdm
from scripts.main_notebook_runner import probe_training_device
from scripts.main_additional_runner import (
    prepare_additional, train_additional, evaluate_grid, status, export_results,
    LEGACY_METHODS, ALL_METHODS, ON_METHODS,
)
PARENT = ROOT / "results" / "main_four_faces_free_30m"
OUTPUT = ROOT / "results" / "main_additional_20261120"
TARGET = 30000
RUN_LEGACY_ROLLOUTS = True
RUN_TRAINING = True
RUN_ADDITIONAL_ROLLOUTS = True
RUN_F_CALIBRATION = True
RUN_DECISION = False
RUN_ALL_CALIBRATION = False
RUN_EXPORT = False


## 共通bankと方式の準備
PARENTには元のbanksとA/B/F各3 seedのmodel_30000.ptが必要です。旧Fや既存のexperiment.jsonは書き換えません。bankの再生成はせず、元ファイルをコピーしてhashを照合します。設定・実装が変わった保存先は拒否します。

In [ ]:
suite = prepare_additional(PARENT, OUTPUT, target=TARGET)
print(f"追加学習: F（距離予算1.5）/ B50（通信半径50m）, seed={suite['training_seeds']}, TARGET={TARGET:,}")
print(f"development: {len(suite['development_ids'])}共通形状 / 保存先: {OUTPUT}")


## 学習デバイスの確認
学習前にCUDAデバイス番号と実際のGeForce製品名を確認します。評価は元実験のevaluation_deviceを引き継ぎ、1条件ずつ実行します。このセルでは学習しません。

In [ ]:
hardware = probe_training_device(OUTPUT / "experiments" / "F" / "settings.yaml")
gpu = hardware['cuda_device_name']
print(f"学習デバイス: {hardware['training_device']} ({gpu})" if gpu else f"学習デバイス: {hardware['training_device']}")
print(f"設定: {hardware['requested_device']} / 評価: {hardware['evaluation_device']}")
print(f"PyTorch: {hardware['torch']} / CUDA: {hardware['torch_cuda']} / CUDA利用可能: {hardware['cuda_available']}")
print(f"Python: {hardware['python']}")


## 既存モデルで未実施のロールアウト
A/B75/HBA/C/D75/HBA_OFF/E/Rを、共通developmentで揃えます。元のcheckpoint・bank・条件が一致する既存結果は来歴を保持して取り込み、未完了分だけ飛ばします。モデルが不足していれば必要なパスを表示して停止します。D75は従来D、HBA_OFFは従来F_OFFです。

In [ ]:
if RUN_LEGACY_ROLLOUTS:
    legacy_rollouts = evaluate_grid(OUTPUT, methods=LEGACY_METHODS, progress_factory=tqdm)
    print("既存方式のdevelopment完了:", legacy_rollouts)


## FとB50の独立学習
seedごとにF→B50を直列実行します。warm-up収集ep、Cost Critic更新、online累積stepと完了ep、development件数をそれぞれtqdm表示します。30k到達episode終端で止まり、元と同じ48形状を評価します。中断後も同じOUTPUT・TARGETで実行します。Replayは変更slotだけを追記し、モデル・optimizer・乱数・cursorと組にして復元します。再開には保存先全体を保持してください。

In [ ]:
if RUN_TRAINING:
    completed_runs = train_additional(OUTPUT, progress_factory=tqdm)
    print("追加学習・development完了:", completed_runs)


## 追加方式のON OFFロールアウト
正規F/F_OFF、B50/D50を共通developmentで評価します。学習時のON評価は取り込み、追加飛行は不足ケースだけです。OFFは同じActorの同じcheckpointを使い、pairwise HOCBFだけを外します。

In [ ]:
if RUN_ADDITIONAL_ROLLOUTS:
    new_rollouts = evaluate_grid(OUTPUT, methods=("F", "F_OFF", "B50", "D50"), progress_factory=tqdm)
    print("追加方式のdevelopment完了:", new_rollouts)


## 正規Fの予算とCritic校正
decisionの固定10形状/N・各3反復で確率方策を評価します。初期sample actionをCriticと初回rolloutで一致させます。学習対象は距離Cost予算1.5で、介入・共通D+Iも診断として残します。予算達成の統計判定は結果解析で行います。

In [ ]:
if RUN_F_CALIBRATION:
    f_calibration = evaluate_grid(OUTPUT, methods=("F",), split="decision", stochastic=True, progress_factory=tqdm)
    print("正規Fの確率評価完了:", f_calibration)


## 任意の追加評価
RUN_DECISION=Trueで全方式のdecision 120形状を評価します。RUN_ALL_CALIBRATION=Trueで学習済みON5方式の確率評価を揃えます。すでに終わったFのケースは再利用します。confirmation_testはこの入口では実行できません。

In [ ]:
if RUN_DECISION:
    decision_rollouts = evaluate_grid(OUTPUT, methods=ALL_METHODS, split="decision", progress_factory=tqdm)
    print("全方式decision完了:", decision_rollouts)
if RUN_ALL_CALIBRATION:
    calibration_rollouts = evaluate_grid(OUTPUT, methods=ON_METHODS, split="decision", stochastic=True, progress_factory=tqdm)
    print("全方式確率評価完了:", calibration_rollouts)


## 途中結果の確認と持ち帰り
完了caseは1本ずつ保存されます。RUN_EXPORT=Trueでbank、条件、学習ログ、小型モデル、生NPZをZIP化します。巨大Replayと完全再開用checkpointは除外しWindows側に保持します。ZIPをMac側へ渡せば保存軌道から解析できます。

In [ ]:
result_status = status(OUTPUT)
for row in result_status:
    print(f"{row['split']}/{row['mode']} {row['method']}/{row['seed']}: {row['episodes']}/{row['expected']} 件, 到達={row['successes']:g}, 違反ep={row['violation_episodes']}")
if RUN_EXPORT:
    archive = export_results(OUTPUT)
    print(f"持ち帰り用ZIP: {archive}")
